# 01 直接建模（3-index CP-SAT）—— 完整数学公式与最优性证书

## 完整模型

**变量**

- $x_{ijk}\in\{0,1\}$：车辆 $k$ 走弧 $(i,j)$，$i,j\in\{0\}\cup C$（$0$=仓库）
- $t_{ik}\in[0,L_0\cdot S]$：车辆 $k$ 在客户 $i$ 的**开始服务时刻**（整数缩放 $S=10000$）
- $q_{ik}\in[0,Q]$：车辆 $k$ 服务完客户 $i$ 后的累计负载

**约束**

1. 每车一个回路：$\text{AddCircuit}(x_{k})$，即
$$\sum_j x_{ijk}=\sum_j x_{jik}=1,\ \forall i,k\qquad(\text{自环允许})$$
2. 每客户恰被一车访问（其余车在该客户自环）：
$$\sum_k x_{iik}=K-1,\ \forall i\in C$$
3. 对称破缺（未使用车辆=后缀，仓库自环 $x_{00k}=1$ 表示车 $k$ 未用）：
$$x_{00k}\le x_{00,k+1}$$
4. 时间窗（允许等待；$M=10^{12}$ 大数，自环 $x_{iik}=1$ 时解除约束）：
$$t_{jk}\ge t_{ik}+s_i S+d_{ij}S-M(1-x_{ijk}),\qquad r_iS\le t_{ik}\le l_iS$$
5. 容量：$q_{jk}\ge q_{ik}+\delta_j-M(1-x_{ijk})$，$0\le q_{ik}\le Q$
6. 返回仓库：$t_{ik}+s_iS+d_{i0}S\le L_0S+M(1-x_{i0k})$
7. 子回路自动消除：弧距离为正 ⇒ 客户环上 $t$ 严格递增矛盾（时间单调性）

**目标**

$$\min\ \sum_{k}\sum_{i\ne j}d_{ij}S\cdot x_{ijk}$$

## 为什么没有"对偶"

- 该模型的回路（AddCircuit）与时间窗约束**没有紧凑的线性化**，LP 松弛/对偶理论不直接适用；
  CP-SAT 用**约束传播 + 分支定界**直接证明最优，**不需要显式对偶**。
- **最优性证书**：求解状态 OPTIMAL ⟺ 可行解（上界）与搜索界（下界）相等：
  $objective = best\_bound = 1918136$（缩放值）。
- 正是这种"无对偶可用"的结构，催生了 02–07 的分解方法（列生成把困难留在定价子问题、
  拉格朗日/LBBD 用乘子与逻辑割构造对偶信息）。


In [1]:
# 环境与演示数据（28 列小池 = 25 条单客户路径 + 3 条最优路线）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
ROUTES = [(13,17,18,19,15,16,14,12), (20,24,25,23,22,21), (5,3,7,8,10,11,9,6,4,2,1)]
pool = [(i,) for i in range(1, n+1)] + ROUTES

def col_cost(p):
    c = 0.0
    prev = 0
    for j in p:
        c += dist(prev, j)
        prev = j
    return c + dist(prev, 0)

def col_mask(p):
    m = 0
    for j in p:
        m |= (1 << j)
    return m

costs = [col_cost(p) for p in pool]
masks = [col_mask(p) for p in pool]
P = len(pool)
print(f"演示池: {P} 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）")


from direct_solve import solve
st, obj, routes, el, bound = solve(3, 120)
print(f"K=3: {st} | 缩放目标 {obj} | 下界(best bound) {bound} | 耗时 {round(el,1)}s")
print("最优性证书: 目标 == 下界 ->", obj == bound and st.name == "OPTIMAL")
exact = 0.0
for r in routes:
    seq = [0] + list(r) + [0]
    c = sum(dist(seq[i], seq[i+1]) for i in range(len(seq)-1))
    exact += c
    print("  路线", r, "距离", round(c, 6))
print("精确总距离", round(exact, 10), "= 191.81 两位小数（BKS）")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
演示池: 28 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）
n=25 KMAX=25 cap=200 总需求=460 LB_K=3


K=3: CpSolverStatus.OPTIMAL | 缩放目标 1918136.0 | 下界(best bound) 1918136.0 | 耗时 0.9s
最优性证书: 目标 == 下界 -> True
  路线 (5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1) 距离 59.488231
  路线 (13, 17, 18, 19, 15, 16, 14, 12) 距离 95.884709
  路线 (20, 24, 25, 23, 22, 21) 距离 36.44068
精确总距离 191.8136197787 = 191.81 两位小数（BKS）
